In [ ]:
#llm chatbot mistral BioBERT
import requests
import json
import os
import numpy as np
import re
import time

try:
    import pdfplumber
except ImportError:
    print('Installing pdfplumber...')
    os.system('pip install pdfplumber')
    import pdfplumber

try:
    from sentence_transformers import models, SentenceTransformer
except ImportError:
    print('Installing sentence-transformers and transformers...')
    os.system('pip install sentence-transformers transformers torch')
    from sentence_transformers import models, SentenceTransformer

# --- Configuration ---
API_KEY = os.environ.get("MISTRAL_API_KEY", "").strip()
if not API_KEY:
    raise SystemExit(
        "Defina a variável de ambiente MISTRAL_API_KEY com a chave da API do Mistral."
    )
MODEL_ID = "mistral-medium"
MISTRAL_API_URL = "https://api.mistral.ai/v1/chat/completions"
PDF_PATH = os.environ.get("SOSORT_PDF_PATH", "data/SOSORT_guideline.pdf")
CHUNK_SIZE = 500
MAX_CONTEXT_LENGTH = 4000  # Max chars in context

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# --- PDF Text Extraction ---
def extract_pdf_text(pdf_path):
    try:
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    except Exception as e:
        print(f"❌ Failed to extract text from PDF: {e}")
        return ""

# --- Smart Sentence-Aware Chunking ---
def smart_chunk_text(text, max_len=CHUNK_SIZE):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current_chunk = [], ""
    for sentence in sentences:
        if len(current_chunk) + len(sentence) < max_len:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# --- Embedding ---
def get_embedding(text, model):
    return model.encode(text, normalize_embeddings=True)

# --- Cosine Similarity ---
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# --- Top-k Retrieval ---
def retrieve_relevant_chunks(chunks, chunk_embeddings, query_embedding, top_k=3):
    similarities = [cosine_similarity(chunk_emb, query_embedding) for chunk_emb in chunk_embeddings]
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    return [chunks[i] for i in top_indices]

# --- Load and Chunk PDF ---
pdf_text = extract_pdf_text(PDF_PATH)
if not pdf_text:
    print("⚠️ No text extracted. Exiting.")
    exit()

chunks = smart_chunk_text(pdf_text)

# --- Load PubMedBERT Embedding Model ---
print("🔬 Loading PubMedBERT model...")
biomedical_model = models.Transformer("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
pooling = models.Pooling(
    word_embedding_dimension=biomedical_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)
embedder = SentenceTransformer(modules=[biomedical_model, pooling])

# --- Embed Chunks with time logging ---
print("⚙️ Embedding PDF chunks (this may take a while)...")
start_time = time.time()
chunk_embeddings = [get_embedding(chunk, embedder) for chunk in chunks]
print(f"✅ Embedding completed in {time.time() - start_time:.2f} seconds.")

# --- RAG Q&A Loop ---
while True:
    user_query = input("\nAsk a question about the PDF (or type 'exit' to quit): ")
    if user_query.strip().lower() in ["exit", "quit", "q"]:
        print("👋 Exiting.")
        break

    user_query_clean = user_query.strip().replace("\n", " ")
    query_embedding = get_embedding(user_query_clean, embedder)
    relevant_chunks = retrieve_relevant_chunks(chunks, chunk_embeddings, query_embedding, top_k=3)
    context = "\n".join(relevant_chunks)

    if len(context) > MAX_CONTEXT_LENGTH:
        context = context[:MAX_CONTEXT_LENGTH]

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Use the following context to answer the user's question."},
        {"role": "system", "content": f"Context from PDF:\n{context}"},
        {"role": "user", "content": user_query}
    ]

    payload = {
        "model": MODEL_ID,
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 1024,
        "stream": False,
        "top_p": 0.95
    }

    try:
        response = requests.post(MISTRAL_API_URL, headers=headers, data=json.dumps(payload))
        response.raise_for_status()
        data = response.json()

        # ✅ Output summarized relevant chunks
        print("\n🔍 Relevant Chunks Preview:")
        for i, chunk in enumerate(relevant_chunks):
            preview = chunk.strip().split(".")[0][:100]
            print(f"{i+1}. {preview}...")

        print("\n🤖 Mistral Answer:\n", data["choices"][0]["message"]["content"])

    except Exception as e:
        print(f"❌ Error during LLM request: {e}")
        print(f"Response: {getattr(response, 'text', None)}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported


🔬 Loading PubMedBERT model...
⚙️ Embedding PDF chunks (this may take a while)...
✅ Embedding completed in 31.61 seconds.
❌ Error during LLM request: 429 Client Error: Too Many Requests for url: https://api.mistral.ai/v1/chat/completions
Response: {"object":"error","message":"Service tier capacity exceeded for this model.","type":"service_tier_capacity_exceeded","param":null,"code":"3505"}

🔍 Relevant Chunks Preview:
1. Recently published high-quality clinical trials on
theeffect of conservative treatment approaches (b...
2. Moreover, disability, pain and HRQoL was demonstrated with
it is also important to consider that bra...
3. The present guidelines include a total of 68
recommendations divided into following topics: bracing ...

🤖 Mistral Answer:
 The provided context does **not** explicitly address the **etiology (causes)** of **idiopathic scoliosis (IS)**—it focuses on **treatment guidelines** (bracing, exercises, respiratory function, etc.). However, based on broader medical lite